# 02 - LoRA on SAM 3 (team stack)
The LoRA adaptation arm on the native `sam3` package, the shared team stack
(`Sam3MedSeg`). Build the model, verify the injection targets, and verify the
trainable-param counts of `finetune_mode="lora"`.

History: this notebook's earlier versions cover the SAM 1 prototype (147K/93.7M,
0.157%), see git history. The HF-transformers verification lives in notebook 03.

**Kernel: select the `sam3-native` env** (the only one with the `sam3` package).

In [ ]:
#autoreload, so edits to .py files show without restarting
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.insert(0, "../src")   #repo convention so `medseg` imports
print(sys.executable)          #sanity check: should be the sam3-native env

## 1. Unit tests: LoRALayer / LinearWithLoRA / LoRA_qkv slice isolation

In [ ]:
!pytest ../tests/test_lora.py -v

## 2. Build the native SAM 3 image model
Loads the gated weights from the HF cache (`load_from_HF=True` default), builds
on CPU (`device='cpu'` default). Takes a minute or two. This is the exact call
inside `Sam3Backbone.__init__`.

In [ ]:
from sam3.model_builder import build_sam3_image_model

model = build_sam3_image_model()
total = sum(p.numel() for p in model.parameters())
print(f"{type(model).__name__}: {total:,} total params")

## 3. LoRA injection targets: 32 fused-qkv attention layers
Native vitdet fuses Q,K,V into ONE `Linear(1024, 3072)` per block (unlike the
HF edition's separate q_proj/v_proj), so we wrap with `LoRA_qkv`, not
`LinearWithLoRA`. Expect 32 blocks, each
`Linear(in_features=1024, out_features=3072, bias=True)`.

In [ ]:
blocks = model.backbone.vision_backbone.trunk.blocks
print(f"{len(blocks)} blocks")
print(blocks[0].attn.qkv)

## 4. The LoRA arm: `Sam3MedSeg(finetune_mode="lora")`
Freeze the 455M-param backbone, inject adapters into all 32 qkv layers, train
adapters + FPN seg head. Expected (verified 2026-07-20):

```
1,066,113 / 455,105,921 trainable  (0.2343%)
  = 524,288 adapters (32 x 2 x 2*4*1024)  +  541,825 FpnSegHead
```

Any other number means the injection state is wrong (e.g. head swept into the
freeze, or a double injection; see docs/lora.md).

In [ ]:
from medseg.models.sam3_medseg import Sam3MedSeg

m = Sam3MedSeg(finetune_mode="lora")
trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
total = sum(p.numel() for p in m.parameters())
print(f"{trainable:,} / {total:,} trainable  ({100*trainable/total:.4f}%)")